# 4.1 — Le neurone convolutif from scratch : kernel C#, puis Conv2d TorchSharp, parité numérique

**Navigation** : [← DataScienceWithAgents (série parente)](../README.md) | [03-Deep-Learning (prérequis)](../03-Deep-Learning/README.md) | [4.2 — ConvNet profonde : pourquoi les résiduelles →](4.2-ConvNet-Profonde-Residuelles.ipynb)

**Kernel** : .NET C# · **Bibliothèques** : TorchSharp, Python.NET et Pillow (prétraitement) · **Niveau** : intermédiaire (post 3.x) · **CPU** : oui

## Pourquoi ce notebook

Le socle [`03-Deep-Learning`](../03-Deep-Learning/README.md) a ouvert la rétropropagation sur des **vecteurs** : un MLP prend des features tabulaires et empile des multiplications matricielles. Le passage à l'image demande un changement de primitive : le neurone convolutif n'opère plus sur des vecteurs, mais sur des **patchs locaux** de l'image — un filtre glisse sur la hauteur et la largeur en partageant ses poids.

Cette transformation semble compliquée (patchs, strides, padding, multi-canal). Elle ne l'est pas : c'est exactement la même primitive — *produit scalaire entre poids et portion locale* — vue sous une forme où les poids sont **partagés spatialement**. La discipline reste la même que dans le 3.1 :

1. **From scratch** : `conv2d` C#, écrit explicitement, gradient vérifié par différence finie.
2. **Framework** : `torch.nn.Conv2d`, mêmes poids initialisés, mêmes entrées.
3. **Parité numérique** : des tolérances explicites adaptées au type et aux données.

L'enjeu pédagogique : montrer qu'un filtre convolutif n'a rien de magique — c'est une multiplication matricielle *localisée*, et la preuve que `np` et `torch` calculent la même formule à une tolérance numérique près rend l'API TorchSharp lisible. Ce notebook installe aussi les briques qui serviront dans le 4.2 (conv profonde, résiduelles) et le 4.3 (transfer learning ResNet).

**Objectifs** : calculer une corrélation, vérifier son gradient et comparer Conv2d ; mesurer translation et pooling. **Prérequis** : tableaux et dérivées. **Durée** : 2–3 heures. [Version Python](4.1-Conv-NumPy-Torch-Allclose.ipynb). Installer TorchSharp-cpu 0.107, Python.NET 3.0.5, CPython et Pillow ; définir `PYTHONNET_PYDLL` et si nécessaire `PYTHONPATH`.


***

## 1. Le problème : un filtre convolutif est un produit scalaire local

**Setup.** On se donne une image `X` de forme `(H, W)` (un seul canal pour commencer). Un filtre convolutif `K` de forme `(kH, kW)` produit une sortie `Y` où chaque pixel `Y[i, j]` est le produit scalaire entre `K` et la fenêtre `X[i:i+kH, j:j+kW]` *centrée* (ou alignée selon la convention — on choisit `valid`, pas de padding ici).

Mathématiquement :
$$ Y[i, j] = \sum_{u=0}^{k_H-1} \sum_{v=0}^{k_W-1} K[u, v] \cdot X[i+u, j+v] $$

Trois observations immédiates :
- Les **mêmes poids** `K` sont utilisés à toutes les positions `(i, j)` → c'est ça le partage spatial.
- Le nombre de multiplications par pixel de sortie est `kH × kW` (pas la taille de l'image).
- Si on empile `C_out` filtres, on obtient une sortie `(H-kH+1, W-kW+1, C_out)`.

**Question.** Les mêmes données et poids donnent-ils des résultats proches entre boucles et vrai `Conv2d` ? Les écarts mesurés seront comparés à des seuils explicites selon le type.


***

## 2. Corrélation valide en C# : multiplication locale explicite


In [1]:
#r "nuget: TorchSharp-cpu, 0.107.0"
#r "nuget: PythonNet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using System.Security.Cryptography;
using System.IO;
using TorchSharp;
using static TorchSharp.torch;
using Python.Runtime;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
torch.set_num_threads(1);
if (!PythonEngine.IsInitialized)
{
    Runtime.PythonDLL = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL") ?? throw new Exception("Définir PYTHONNET_PYDLL");
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}

using (Py.GIL())
{
    dynamic sys = Py.Import("sys"), pil = Py.Import("PIL");
    Console.WriteLine($".NET {Environment.Version} | TorchSharp {typeof(Tensor).Assembly.GetName().Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | CPython {sys.version} | Pillow {pil.__version__}");
}

record PlotSvg(string Svg);
Formatter.Register<PlotSvg>((p, w) => w.Write(p.Svg), "text/html");
string F(double x) => x.ToString("0.###", CultureInfo.InvariantCulture);
double[, ] Conv(double[, ] x, double[, ] k)
{
    int h = x.GetLength(0), w = x.GetLength(1), kh = k.GetLength(0), kw = k.GetLength(1);
    if (kh < 1 || kw < 1 || kh > h || kw > w || x.Cast<double>().Concat(k.Cast<double>()).Any(v => !double.IsFinite(v)))
        throw new ArgumentException("Dimensions ou valeurs invalides");
    var y = new double[h - kh + 1, w - kw + 1];
    for (int i = 0; i < y.GetLength(0); i++)
        for (int j = 0; j < y.GetLength(1); j++)
            for (int u = 0; u < kh; u++)
                for (int v = 0; v < kw; v++)
                    y[i, j] += x[i + u, j + v] * k[u, v];
    return y;
}

double Diff(double[, ] a, double[, ] b)
{
    if (a.GetLength(0) != b.GetLength(0) || a.GetLength(1) != b.GetLength(1))
        throw new Exception("forme");
    double e = 0;
    for (int i = 0; i < a.GetLength(0); i++)
        for (int j = 0; j < a.GetLength(1); j++)
            e = Math.Max(e, Math.Abs(a[i, j] - b[i, j]));
    return e;
}

double[, ] RandomArray(int h, int w, int seed)
{
    var r = new Random(seed);
    var a = new double[h, w];
    for (int i = 0; i < h; i++)
        for (int j = 0; j < w; j++)
            a[i, j] = Math.Sqrt(-2 * Math.Log(1 - r.NextDouble())) * Math.Cos(2 * Math.PI * r.NextDouble());
    return a;
}

var X = RandomArray(8, 8, 42);
var K = RandomArray(3, 3, 43);
var Y = Conv(X, K);
Console.WriteLine($"X 8×8, K 3×3 -> Y {Y.GetLength(0)}×{Y.GetLength(1)}, Y[0,0]={Y[0, 0]:F10}, Y[1,2]={Y[1, 2]:F10}");
double direct = 0;
for (int u = 0; u < 3; u++)
    for (int v = 0; v < 3; v++)
        direct += X[u, v] * K[u, v];
Console.WriteLine($"Vérification directe : écart {Math.Abs(direct - Y[0, 0]):E3}");


Installed Packages PythonNet, 3.0.5 TorchSharp-cpu, 0.107.0

.NET 10.0.14 | TorchSharp 0.107.0.0 | Python.NET 3.0.5.0 | CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | Pillow 13.0.0


X 8×8, K 3×3 -> Y 6×6, Y[0,0]=-1.6896534405, Y[1,2]=-3.6433047365


Vérification directe : écart 0.000E+000


**Lecture.** La boucle calcule la corrélation sans retourner le noyau ; la sortie 6×6 est entièrement interne à l’image. Les générateurs aléatoires C# et NumPy diffèrent.


**Question.** La boucle est claire mais coûteuse ; une contraction sur fenêtres `unfold` et `matmul` illustre une autre voie de calcul.


In [2]:
Console.WriteLine("Contraction des fenêtres par matmul");
var xt = torch.tensor(X, dtype: ScalarType.Float64).reshape(1, 1, 8, 8);
var kt = torch.tensor(K, dtype: ScalarType.Float64).reshape(9, 1);
var windows = xt.unfold(2, 3, 1).unfold(3, 3, 1);
var rows = windows.permute(0, 2, 3, 1, 4, 5).contiguous().reshape(36, 9);
var contracted = rows.matmul(kt).reshape(6, 6);
var vec = contracted.data<double>().ToArray();
double err = Y.Cast<double>().Zip(vec, (a, b) => Math.Abs(a - b)).Max();
Console.WriteLine($"unfold + matmul : 6×6 ; écart maximal {err:E3}, seuil 1e-12 ; {(err < 1e-12 ? "PASS" : "FAIL")}");


Contraction des fenêtres par matmul


unfold + matmul : 6×6 ; écart maximal 8.882E-016, seuil 1e-12 ; PASS



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture.** Les fenêtres sont obtenues par `unfold` puis contractées par `matmul`. L’écart affiché mesure les effets de l’ordre flottant et ne préjuge pas du backend choisi par Conv2d.


***

## 3. Rétropropagation : le gradient d'un filtre convolutif, vérifié par différence finie

**Gradient des poids.** Chaque coefficient accumule les produits de l’entrée par le gradient de sortie. L’adjoint relatif à l’entrée est la convolution transposée ; il possède une autre forme.


In [3]:
double[, ] Backward(double[, ] dy, double[, ] x, int kh, int kw)
{
    var g = new double[kh, kw];
    for (int u = 0; u < kh; u++)
        for (int v = 0; v < kw; v++)
            for (int i = 0; i < dy.GetLength(0); i++)
                for (int j = 0; j < dy.GetLength(1); j++)
                    g[u, v] += dy[i, j] * x[i + u, j + v];
    return g;
}

double Loss(double[, ] k) => Conv(X, k).Cast<double>().Sum(v => v * v);
var dy = new double[6, 6];
for (int i = 0; i < 6; i++)
    for (int j = 0; j < 6; j++)
        dy[i, j] = 2 * Y[i, j];
var grad = Backward(dy, X, 3, 3);
double ga = 0, gr = 0;
for (int u = 0; u < 3; u++)
    for (int v = 0; v < 3; v++)
    {
        var kp = (double[, ])K.Clone();
        var km = (double[, ])K.Clone();
        kp[u, v] += 1e-5;
        km[u, v] -= 1e-5;
        double n = (Loss(kp) - Loss(km)) / 2e-5;
        ga = Math.Max(ga, Math.Abs(n - grad[u, v]));
        gr = Math.Max(gr, Math.Abs(n - grad[u, v]) / (Math.Abs(n) + 1e-12));
    }

var cv = torch.nn.Conv2d(1, 1, 3, bias: false, dtype: ScalarType.Float64);
using (var ng = torch.no_grad())
    cv.weight.copy_(torch.tensor(K, dtype: ScalarType.Float64).reshape(1, 1, 3, 3));
var outNative = cv.forward(xt);
var loss = outNative.pow(2).sum();
loss.backward();
double gn = cv.weight.grad.data<double>().ToArray().Zip(grad.Cast<double>(), (a, b) => Math.Abs(a - b)).Max();
Console.WriteLine($"Gradient poids : différences centrales absolue {ga:E3}, relative {gr:E3} ; backward natif {gn:E3} ; {(ga < 1e-5 ? "PASS" : "FAIL")}");


Gradient poids : différences centrales absolue 8.867E-009, relative 7.685E-009 ; backward natif 2.842E-014 ; PASS



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture.** La différence centrale teste les neuf coefficients et le backward natif fournit une comparaison indépendante ; le seuil absolu est 1e-5.


***

## 4. Conv2d multi-canal et parité numérique C# ↔ TorchSharp Conv2d

**Généralisation.** Avec trois canaux et quatre filtres, TorchSharp attend `(C_out,C_in,kH,kW)`. Une boucle indépendante additionne les produits locaux sur les canaux.


In [4]:
var rng = new Random(42);
double Normal() => Math.Sqrt(-2 * Math.Log(1 - rng.NextDouble())) * Math.Cos(2 * Math.PI * rng.NextDouble());
double[] xm = Enumerable.Range(0, 3 * 12 * 12).Select(_ => Normal()).ToArray();
// Stockage source (Cout,kH,kW,Cin), puis permutation explicite pour TorchSharp.
double[] sourceK = Enumerable.Range(0, 4 * 3 * 3 * 3).Select(_ => Normal()).ToArray();
double[] km = new double[sourceK.Length];
for (int co = 0; co < 4; co++)
    for (int u = 0; u < 3; u++)
        for (int v = 0; v < 3; v++)
            for (int ci = 0; ci < 3; ci++)
                km[((co * 3 + ci) * 3 + u) * 3 + v] = sourceK[((co * 3 + u) * 3 + v) * 3 + ci];
Console.WriteLine("Poids source (4,3,3,3) : ordre Cout,kH,kW,Cin → Cout,Cin,kH,kW");
double[] manual = new double[4 * 10 * 10];
for (int co = 0; co < 4; co++)
    for (int ci = 0; ci < 3; ci++)
        for (int i = 0; i < 10; i++)
            for (int j = 0; j < 10; j++)
                for (int u = 0; u < 3; u++)
                    for (int v = 0; v < 3; v++)
                        manual[(co * 10 + i) * 10 + j] += xm[(ci * 12 + i + u) * 12 + j + v] * km[((co * 3 + ci) * 3 + u) * 3 + v];
foreach (var dtype in new[]
{
    ScalarType.Float32,
    ScalarType.Float64
}

)
{
    var tx = torch.tensor(xm, dtype: ScalarType.Float64).to_type(dtype).reshape(1, 3, 12, 12);
    var tw = torch.tensor(km, dtype: ScalarType.Float64).to_type(dtype).reshape(4, 3, 3, 3);
    var layer = torch.nn.Conv2d(3, 4, 3, bias: false, dtype: dtype);
    using (var ng = torch.no_grad())
        layer.weight.copy_(tw);
    var result = layer.forward(tx);
    double e = result.to_type(ScalarType.Float64).data<double>().ToArray().Zip(manual, (a, b) => Math.Abs(a - b)).Max();
    double tol = dtype == ScalarType.Float32 ? 1e-5 : 1e-10;
    Console.WriteLine($"Conv2d {dtype} : entrée (1,3,12,12), poids (4,3,3,3), sortie (1,4,10,10), écart {e:E3}, seuil {tol:E0} ; {(e < tol ? "PASS" : "FAIL")}");
}


Poids source (4,3,3,3) : ordre Cout,kH,kW,Cin → Cout,Cin,kH,kW


Conv2d Float32 : entrée (1,3,12,12), poids (4,3,3,3), sortie (1,4,10,10), écart 2.473E-006, seuil 1E-005 ; PASS


Conv2d Float64 : entrée (1,3,12,12), poids (4,3,3,3), sortie (1,4,10,10), écart 3.553E-015, seuil 1E-010 ; PASS



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture.** Les écarts f32 et f64 affichés concernent ces données, poids et backend. Les seuils 1e-5 et 1e-10 ne sont pas des epsilons machine universels.

La boucle manuelle multicanal est calculée en double et sert de référence aux deux moteurs ; l’écart f32 inclut la conversion des entrées et poids ainsi que les arrondis natifs. Le carnet Python compare aussi une boucle manuelle f32.

Les mesures affichées sont **2,473e-6** en float32 et **3,553e-15** en float64. Pour le second régime, entrées, poids et module utilisent la double précision ; convertir seulement la sortie ne restituerait pas la précision perdue pendant les produits. L’epsilon machine est un espacement relatif près de 1, tandis que l’écart absolu dépend aussi de l’échelle et de l’accumulation des termes. Les deux seuils contrôlent ce banc sans imposer une égalité bit à bit à tout backend.

Les poids pédagogiques suivent l’ordre `(C_out,kH,kW,C_in)` ; Conv2d attend `(C_out,C_in,kH,kW)`. Cette permutation conserve le rôle de chaque coefficient et le partage spatial des filtres.


***

## 5. Filtres appris vs filtres artisanaux : contours et texture sur une image réelle

**Image réelle.** On charge une image en niveaux de gris : une photographie de chat de Wikimedia Commons (`File:Cat03.jpg`, auteur Fir0002, licences **GFDL 1.2** et **CC BY-NC 3.0**, attribution requise). Elle est livrée avec le carnet sous forme d'instantané figé dans `assets/cat03-120px.jpg` — les sorties ci-dessous ne dépendent donc ni du réseau ni de la version servie un jour donné. La classe 281 d'ImageNet (« tabby ») est celle que ce type d'image porte dans un jeu étiqueté ; elle sert ici de repère, pas de licence. Le chargement, la conversion en gris et le redimensionnement sont faits par Pillow via Python.NET ; le calcul reste en C#. On applique trois filtres :
- **Sobel vertical** : bords francs (transitions horizontales de luminance)
- **Sobel horizontal** : bords francs (transitions verticales)
- **Gaussien 5×5** : flou, moyenne locale

C'est ce que fait un CNN *au début* d'un réseau profond (VGG, ResNet) : ses premiers filtres convergent vers des détecteurs de contours/texture très proches de Sobel et Gaussien, appris à partir de rien.

In [5]:
string Panels(string title, params (string, double[, ])[] panels)
{
    var sb = new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='1040' height='300'><rect width='1040' height='300' fill='white'/><text x='520' y='20' text-anchor='middle' font-family='sans-serif'>{System.Net.WebUtility.HtmlEncode(title)}</text>");
    for (int n = 0; n < panels.Length; n++)
    {
        var(name, a) = panels[n];
        int h = a.GetLength(0), w = a.GetLength(1);
        double lo = a.Cast<double>().Min(), hi = a.Cast<double>().Max();
        sb.Append($"<text x='{130 + n * 260}' y='42' text-anchor='middle' font-size='12'>{System.Net.WebUtility.HtmlEncode(name)}</text>");
        for (int i = 0; i < h; i++)
            for (int j = 0; j < w; j++)
            {
                int c = (int)(255 * (a[i, j] - lo) / Math.Max(1e-12, hi - lo));
                sb.Append($"<rect x='{F(25 + n * 260 + j * 210.0 / w)}' y='{F(50 + i * 210.0 / h)}' width='{F(210.1 / w)}' height='{F(210.1 / h)}' fill='rgb({c},{c},{c})'/>");
            }

        sb.Append($"<text x='{130 + n * 260}' y='280' text-anchor='middle' font-size='10'>colonne (pixels), 0–{w - 1}</text><text transform='translate({15 + n * 260} 155) rotate(-90)' text-anchor='middle' font-size='10'>ligne 0–{h - 1}</text>");
    }

    return sb + "</svg>";
}

var sobel = new double[, ]
{
    {
        -1,
        0,
        1
    },
    {
        -2,
        0,
        2
    },
    {
        -1,
        0,
        1
    }
};
string path = Path.Combine("assets", "cat03-120px.jpg");
string hash = Convert.ToHexString(SHA256.HashData(File.ReadAllBytes(path))).ToLowerInvariant();
if (hash != "b04b15ce638226f5d8a081e559539efdcb8bed0fb67d35de5669743366605990")
    throw new Exception("Image altérée");
double[, ] cat = new double[96, 96];
using (Py.GIL())
{
    dynamic pil = Py.Import("PIL.Image"), np = Py.Import("numpy");
    using PyObject image = pil.open(path);
    using PyObject gray = ((dynamic)image).convert("L");
    var size = new PyTuple(new PyObject[] { new PyInt(96), new PyInt(96) });
    using PyObject resized = ((dynamic)gray).resize(size);
    using PyObject pixels = np.asarray(resized, dtype: np.float64);
    double[] flat = ((dynamic)pixels).flatten().tolist().As<double[]>();
    for (int i = 0; i < 96; i++)
        for (int j = 0; j < 96; j++)
            cat[i, j] = flat[i * 96 + j] / 255;
}

var sh = new double[3, 3];
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 3; j++)
        sh[i, j] = sobel[j, i];
var gauss = new double[5, 5];
double sum = 0;
for (int i = 0; i < 5; i++)
    for (int j = 0; j < 5; j++)
    {
        gauss[i, j] = Math.Exp(-((i - 2) * (i - 2) + (j - 2) * (j - 2)) / 2.0);
        sum += gauss[i, j];
    }

for (int i = 0; i < 5; i++)
    for (int j = 0; j < 5; j++)
        gauss[i, j] /= sum;
var yv = Conv(cat, sobel);
var yh = Conv(cat, sh);
var yg = Conv(cat, gauss);
Console.WriteLine($"Image figée : {new FileInfo(path).Length} octets, sha256 {hash}, pixels 96×96 somme {cat.Cast<double>().Sum():F9}");
foreach (var(name, k, reference)in new[]
{
    ("vertical", sobel, yv),
    ("horizontal", sh, yh)
}

)
{
    var layer = torch.nn.Conv2d(1, 1, 3, bias: false, dtype: ScalarType.Float32);
    using (var ng = torch.no_grad())
        layer.weight.copy_(torch.tensor(k, dtype: ScalarType.Float32).reshape(1, 1, 3, 3));
    var xcat = torch.tensor(cat, dtype: ScalarType.Float32).reshape(1, 1, 96, 96);
    var actual = layer.forward(xcat);
    double e = actual.to_type(ScalarType.Float64).data<double>().ToArray().Zip(reference.Cast<double>(), (a, b) => Math.Abs(a - b)).Max();
    Console.WriteLine($"Sobel {name} Conv2d f32 : écart {e:E3}");
}

display(new PlotSvg(Panels("Filtres sur le chat", ("Image", cat), ("Sobel vertical", yv), ("Sobel horizontal", yh), ("Gaussien", yg))));


Image figée : 5309 octets, sha256 b04b15ce638226f5d8a081e559539efdcb8bed0fb67d35de5669743366605990, pixels 96×96 somme 5557.509803922


Sobel vertical Conv2d f32 : écart 2.487E-007


Sobel horizontal Conv2d f32 : écart 3.527E-007


Filtres sur le chat Image <rect x='138.75' y='104.688' width='2.189' height='2.189' fill='rg


warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture.** Les filtres de Sobel révèlent les changements de luminance ; le Gaussien lisse les détails. Les écarts aux vrais Conv2d f32 sont mesurés ci-dessus et dépendent du type et du backend. Chaque panneau étire séparément ses niveaux de gris : noir correspond à son minimum et blanc à son maximum ; les amplitudes ne se comparent donc pas directement entre panneaux.


## Exercice 1 : Laplacien

Implémentez `Laplacien` : retournez la carte manuelle et le vrai module `Conv2d` double pour le noyau `[[0,1,0],[1,-4,1],[0,1,0]]`. Comparez-les visuellement à Sobel. La fixture contrôle les formes, valeurs finies et cartes à 1e-10.


In [6]:
// Exercice 1 : carte manuelle et vrai module Conv2d double.
(double[, ], TorchSharp.Modules.Conv2d)? Laplacien(double[, ] image) => null; // TODO
var ex1 = Laplacien(cat);
if (ex1 == null)
    Console.WriteLine("Exercice 1 à compléter");
else
    try
    {
        bool ok = true;
        var lap = new double[, ]
        {
            {
                0,
                1,
                0
            },
            {
                1,
                -4,
                1
            },
            {
                0,
                1,
                0
            }
        };
        foreach (var image in new[]
        {
            cat,
            RandomArray(7, 9, 72)
        }

        )
        {
            var answer = Laplacien(image);
            if (answer == null)
            {
                ok = false;
                continue;
            }

            var(manual1, module) = answer.Value;
            var expected = Conv(image, lap);
            int height = image.GetLength(0), width = image.GetLength(1), oh = height - 2, ow = width - 2;
            ok &= module != null && module.weight.dtype == ScalarType.Float64 && module.weight.shape.SequenceEqual(new long[] { 1, 1, 3, 3 });
            if (module == null)
                continue;
            ok &= module.weight.data<double>().ToArray().Zip(lap.Cast<double>(), (a, b) => Math.Abs(a - b)).Max() < 1e-10;
            var actual = module.forward(torch.tensor(image, dtype: ScalarType.Float64).reshape(1, 1, height, width)).data<double>().ToArray();
            ok &= manual1.GetLength(0) == oh && manual1.GetLength(1) == ow && manual1.Cast<double>().All(double.IsFinite) && actual.Length == oh * ow && actual.All(double.IsFinite) && Diff(manual1, expected) < 1e-10 && actual.Zip(expected.Cast<double>(), (a, b) => Math.Abs(a - b)).Max() < 1e-10;
        }

        Console.WriteLine(ok ? "Fixtures Laplacien validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }


Exercice 1 à compléter



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



***

## 6. Pooling et équivariance : ce que la convolution *fait* aux features

**Question.** Pourquoi la convolution est-elle équivariante par translation ? Si on translate l'image d'un pixel, la sortie du filtre devrait translater d'autant (pour un filtre qui détecte une caractéristique locale : un bord horizontal reste un bord horizontal, peu importe sa position verticale).

Démonstration : on translate une image jouet (un bord franc), on observe que la sortie du filtre (Sobel vertical, qui détecte des changements horizontaux) suit la translation d'**un pixel près** — c'est l'équivariance sur la zone intérieure, sous réserve des bords.

In [7]:
double[, ] Shift(double[, ] a, int t)
{
    var b = new double[a.GetLength(0), a.GetLength(1)];
    for (int i = 0; i < a.GetLength(0); i++)
        for (int j = t; j < a.GetLength(1); j++)
            b[i, j] = a[i, j - t];
    return b;
}

(double, int, int) MaxAt(double[, ] a)
{
    double max = double.NegativeInfinity;
    int ri = 0, rj = 0;
    for (int i = 0; i < a.GetLength(0); i++)
        for (int j = 0; j < a.GetLength(1); j++)
            if (a[i, j] > max)
            {
                max = a[i, j];
                ri = i;
                rj = j;
            }

    return (max, ri, rj);
}

var img = new double[20, 20];
for (int i = 0; i < 20; i++)
    for (int j = 10; j < 20; j++)
        img[i, j] = 1;
var shifted = Shift(img, 3);
var yo = Conv(img, sobel);
var ys = Conv(shifted, sobel);
var po = MaxAt(yo);
var ps = MaxAt(ys);
double inside = 0;
for (int i = 0; i < 18; i++)
    for (int j = 3; j < 18; j++)
        inside = Math.Max(inside, Math.Abs(ys[i, j] - yo[i, j - 3]));
Console.WriteLine($"Cartes 18×18 : max colonnes {po.Item3} et {ps.Item3}, décalage {ps.Item3 - po.Item3}; erreur intérieure {inside:E3}");
display(new PlotSvg(Panels("Translation et filtre Sobel", ("Image", img), ("Sobel", yo), ("Image translatée", shifted), ("Sobel translaté", ys))));


Cartes 18×18 : max colonnes 8 et 11, décalage 3; erreur intérieure 0.000E+000


Translation et filtre Sobel Image colonne (pixels), 0–19 ligne 0–19 Sobel colonne (pixels), 0–17 ligne 0–17 Image translatée colonne (pixels), 0–19 ligne 0–19 Sobel translaté colonne (pixels), 0–17 ligne 0–17

**Lecture.** Le maximum suit le bord, et l’écart sur la zone intérieure compare les cartes. Les bords imposent une convention ; cette propriété est une équivariance, pas une invariance de la carte. Chaque panneau étire séparément ses niveaux de gris : noir correspond à son minimum et blanc à son maximum ; les amplitudes ne se comparent donc pas directement entre panneaux.


In [8]:
double[, ] Pool(double[, ] a)
{
    var b = new double[a.GetLength(0) / 2, a.GetLength(1) / 2];
    for (int i = 0; i < b.GetLength(0); i++)
        for (int j = 0; j < b.GetLength(1); j++)
            b[i, j] = Math.Max(Math.Max(a[2 * i, 2 * j], a[2 * i + 1, 2 * j]), Math.Max(a[2 * i, 2 * j + 1], a[2 * i + 1, 2 * j + 1]));
    return b;
}

var p0 = Pool(yo);
var p1 = Pool(Conv(Shift(img, 1), sobel));
Console.WriteLine($"Pooling 2×2 : maxima colonnes {MaxAt(p0).Item3}, {MaxAt(p1).Item3} ; différence cartes {Diff(p0, p1):G6}");
var impulse = new double[4, 4];
impulse[0, 1] = 1;
Console.WriteLine($"Contre-exemple de phase : différence pooling {Diff(Pool(impulse), Pool(Shift(impulse, 1)))}");


Pooling 2×2 : maxima colonnes 4, 4 ; différence cartes 4


Contre-exemple de phase : différence pooling 1


**Lecture.** Les positions des maxima peuvent coïncider alors que les cartes diffèrent. L’impulsion franchissant une frontière de fenêtre montre la sensibilité du pooling à la phase.

Une petite fixture rend la phase visible : avec des fenêtres disjointes de deux colonnes, `[1,0,0,0]` et son décalage `[0,1,0,0]` donnent tous deux `[1,0]` après max-pooling ; le décalage suivant `[0,0,1,0]` donne `[0,1]`. Le maximum reste donc parfois dans la même fenêtre, puis en franchit la frontière. En deux dimensions, le raisonnement s’applique à chaque axe des fenêtres `2×2`.

Une carte convolutive équivariante se déplace avec l’entrée sur la zone intérieure ; une représentation invariante resterait identique. Le sous-échantillonnage peut rendre certaines petites translations moins visibles, mais sa phase et les bords empêchent d’en déduire un rayon d’invariance universel, même en empilant plusieurs couches.


## Exercice 2 : huit filtres

Implémentez `HuitCartes` pour les huit filtres fournis, puis affichez une grille 2×4 et discutez leurs activations. Les fixtures contrôlent chaque carte.


In [9]:
// Exercice 2 : huit filtres fournis et leurs huit cartes, grille 2×4 à afficher.
double[][, ] HuitCartes(double[, ] image, double[][, ] filters) => null; // TODO
var filters = Enumerable.Range(0, 8).Select(i => RandomArray(3, 3, 42 + i)).ToArray();
var ex2 = HuitCartes(cat, filters);
if (ex2 == null)
    Console.WriteLine("Exercice 2 à compléter");
else
    try
    {
        bool ok = true;
        foreach (var image in new[]
        {
            cat,
            RandomArray(7, 9, 71)
        }

        )
        {
            var cards = HuitCartes(image, filters);
            ok &= cards != null && cards.Length == 8;
            if (cards == null || cards.Length != 8)
                continue;
            for (int i = 0; i < 8; i++)
                ok &= cards[i].GetLength(0) == image.GetLength(0) - 2 && cards[i].GetLength(1) == image.GetLength(1) - 2 && cards[i].Cast<double>().All(double.IsFinite) && Diff(cards[i], Conv(image, filters[i])) < 1e-10;
        }

        Console.WriteLine(ok ? "Fixtures huit filtres validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }


Exercice 2 à compléter


## À retenir

La corrélation calcule des produits locaux partagés ; son gradient de poids accumule les contributions de chaque fenêtre. Les comparaisons de précision et les effets de bord doivent être mesurés. La translation déplace la carte intérieure ; le pooling dépend de sa phase.

Poursuivez avec [4.2 — ConvNet profonde](4.2-ConvNet-Profonde-Residuelles.ipynb).


## Exercice 3 : translations et pooling

Implémentez `Positions` : pour les translations 0 à 5, retournez les colonnes des maxima avant et après pooling. Tracez les deux séries et discutez phase et bords ; aucun seuil universel n’est supposé.


In [10]:
// Exercice 3 : colonnes des maxima avant et après pooling pour les translations 0 à 5.
(int[], int[])? Positions(double[, ] image, double[, ] filtre) => null; // TODO
var ex3 = Positions(cat, sobel);
if (ex3 == null)
    Console.WriteLine("Exercice 3 à compléter");
else
    try
    {
        bool ok = true;
        foreach (var image in new[]
        {
            cat,
            RandomArray(12, 13, 73)
        }

        )
        {
            var answer = Positions(image, sobel);
            if (answer == null)
            {
                ok = false;
                continue;
            }

            var(a, b) = answer.Value;
            ok &= a.Length == 6 && b.Length == 6;
            for (int t = 0; t < 6 && a.Length == 6 && b.Length == 6; t++)
            {
                var card = Conv(Shift(image, t), sobel);
                ok &= a[t] == MaxAt(card).Item3 && b[t] == MaxAt(Pool(card)).Item3;
            }
        }

        Console.WriteLine(ok ? "Fixtures translations validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }


Exercice 3 à compléter


***

## References

1. Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, chap. 9 (Convolutional Networks). MIT Press.
2. He, K., Zhang, X., Ren, S. & Sun, J. (2015). *Deep Residual Learning for Image Recognition*. arXiv:1512.03385.
3. Krizhevsky, A., Sutskever, I. & Hinton, G. (2012). *ImageNet Classification with Deep Convolutional Neural Networks* (AlexNet). NIPS 2012.
4. LeCun, Y. et al. (1998). *Gradient-Based Learning Applied to Document Recognition* (LeNet-5). Proceedings of the IEEE.
5. Paszke, A. et al. (2019). *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. NeurIPS 2019.
6. PIL (Python Imaging Library) — utilisé pour charger l'image du chat.
7. Wikimedia Commons — image du chat (`Cat03.jpg`, licences GFDL 1.2 et CC BY-NC 3.0, auteur Fir0002 / Flagstaffotos ;
   attribution requise, et la clause NonCommercial borne la redistribution).
## Vérifiez votre compréhension

<details><summary>Pourquoi distinguer équivariance et invariance ?</summary>Une carte équivariante se déplace avec l’entrée ; une carte invariante reste inchangée.</details>

<details><summary>Pourquoi deux tolérances ?</summary>Les types et les ordres de calcul produisent des arrondis différents.</details>

## Pour aller plus loin

Étudier les convolutions avec padding et stride et leurs effets aux frontières.
